# Lumbar Spine MRI Analysis with MedGemma

This notebook demonstrates how to work with **DICOM medical images** from the [RSNA 2024 Lumbar Spine Degenerative Classification](https://www.kaggle.com/competitions/rsna-2024-lumbar-spine-degenerative-classification) competition and use a locally hosted **MedGemma-4b** model (via LM Studio) for AI-assisted radiological analysis.

## Learning objectives
1. Understand the **DICOM** image format used in clinical radiology
2. Load, inspect, and visualize **lumbar spine MRI** series (Sagittal T1, Sagittal T2/STIR, Axial T2)
3. Perform window/level normalization to display MRI contrast correctly
4. Build multimodal prompts and call the **MedGemma** API for radiological interpretation
5. Compare AI analysis across different MRI series types

## Clinical context
The five degenerative conditions evaluated in this competition are:

| Condition | What it means |
| :--- | :--- |
| **Spinal Canal Stenosis** | Narrowing of the central canal compressing the spinal cord/cauda equina |
| **Left Neural Foraminal Narrowing** | Left exit channel for nerve roots is compressed |
| **Right Neural Foraminal Narrowing** | Right exit channel for nerve roots is compressed |
| **Left Subarticular Stenosis** | Left lateral recess narrowing compressing nerve roots |
| **Right Subarticular Stenosis** | Right lateral recess narrowing compressing nerve roots |

Each condition is assessed at five intervertebral disc levels: **L1/L2, L2/L3, L3/L4, L4/L5, L5/S1**, with severity graded as **Normal/Mild**, **Moderate**, or **Severe**.

---
## Data download
Download the competition dataset from Kaggle (requires acceptance of competition rules):
```bash
# Install Kaggle CLI if needed
pip install kaggle

# Place your kaggle.json API key in ~/.kaggle/kaggle.json, then:
kaggle competitions download -c rsna-2024-lumbar-spine-degenerative-classification -p ./data/rsna-2024-lumbar-spine/
cd ./data/rsna-2024-lumbar-spine/ && unzip rsna-2024-lumbar-spine-degenerative-classification.zip
```
Expected directory layout after extraction:
```
data/rsna-2024-lumbar-spine/
├── train.csv
├── train_label_coordinates.csv
├── train_series_descriptions.csv
└── train_images/
    └── <study_id>/
        └── <series_id>/
            └── <instance_number>.dcm
```

## 🔧 Install dependencies

Run this cell once to install all required packages into the current kernel.

In [ ]:
# Uncomment and run if packages are not yet installed
# %pip install pydicom pillow matplotlib numpy pandas requests ipywidgets

## 📦 Imports

In [ ]:
import base64
import io
import mimetypes
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import pydicom
import requests
from PIL import Image
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    widgets_available = True
except ImportError:
    widgets_available = False

print(f"pydicom version : {pydicom.__version__}")
print(f"Pillow version  : {Image.__version__}")
print(f"Widgets available: {widgets_available}")

## ⚙️ Runtime configuration

The notebook calls the **LM Studio OpenAI-compatible endpoint** running on your Windows 11 host.
Set `LM_STUDIO_BASE_URL` to match your Windows host IP as seen from WSL (typically the gateway address).

```bash
# Find Windows host IP from WSL:
cat /etc/resolv.conf | grep nameserver
# or
ip route | grep default
```
Override these defaults by setting environment variables before launching Jupyter:
```bash
export LM_STUDIO_BASE_URL=http://192.168.1.2:1234/v1
export LM_STUDIO_MODEL=medgemma-4b-it
```

In [ ]:
LM_STUDIO_BASE_URL = os.getenv("LM_STUDIO_BASE_URL", "http://192.168.1.2:1234/v1")
LM_STUDIO_MODEL    = os.getenv("LM_STUDIO_MODEL",    "medgemma-4b-it")

# Root folder of the RSNA dataset
RSNA_DATA_ROOT = Path("./data/rsna-2024-lumbar-spine")

print(f"LM Studio endpoint : {LM_STUDIO_BASE_URL}")
print(f"Model              : {LM_STUDIO_MODEL}")
print(f"RSNA data root     : {RSNA_DATA_ROOT.resolve()}")
print(f"Data root exists   : {RSNA_DATA_ROOT.exists()}")

---

## 🗂️ What is DICOM?

**DICOM** (Digital Imaging and Communications in Medicine) is the universal standard for storing, transmitting, and displaying medical imaging data. Every MRI, CT scan, or X-ray produced in a modern hospital is stored as a DICOM file.

### Why DICOM instead of JPEG/PNG?

| Property | JPEG/PNG | DICOM (.dcm) |
| :--- | :--- | :--- |
| **Bit depth** | 8-bit (256 levels) | 12–16-bit (4,096–65,536 levels) |
| **Metadata** | Minimal (EXIF) | Rich: patient ID, acquisition params, orientation |
| **Multi-frame** | No | Yes — a single file can hold a 3D volume |
| **Coordinate system** | Pixel-only | Physical world coordinates (mm) |
| **Clinical use** | Never used in radiology | Mandatory standard globally |

### Key DICOM metadata fields

```
StudyInstanceUID      → unique identifier for the entire imaging study
SeriesInstanceUID     → unique identifier for one series of images
SOPInstanceUID        → unique identifier for a single image slice
Modality              → MR, CT, CR, DX, …
SeriesDescription     → e.g. "Sagittal T2/STIR", "Axial T2"
ImagePositionPatient  → 3D position (x, y, z) in mm of the first voxel
ImageOrientationPatient → direction cosines defining the image plane
PixelSpacing          → physical size of each pixel in mm
RescaleIntercept/Slope → converts stored integers → Hounsfield Units (CT) or a.u. (MR)
WindowCenter/Width    → radiologist's display window preset
```

### MRI series used in lumbar spine studies

| Series | Plane | Best for |
| :--- | :--- | :--- |
| **Sagittal T1** | Side view | Vertebral body height, fatty marrow, alignment |
| **Sagittal T2/STIR** | Side view | Disc hydration, spinal canal, cord signal |
| **Axial T2** | Top-down cross-section | Foraminal narrowing, lateral recess, central canal |

> In T2-weighted MRI, **fluid appears bright** (white) — healthy discs are bright because they contain water-rich nucleus pulposus. Degenerated discs appear dark.

---

## 📋 Explore the dataset metadata

In [ ]:
def load_csv_if_exists(path: Path, label: str) -> pd.DataFrame | None:
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{'='*60}")
        print(f"  {label}  ({path.name})")
        print(f"{'='*60}")
        print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        display(df.head(3))
        return df
    else:
        print(f"[MISSING] {path} — download the dataset first.")
        return None


df_train       = load_csv_if_exists(RSNA_DATA_ROOT / "train.csv",                    "Train labels")
df_series      = load_csv_if_exists(RSNA_DATA_ROOT / "train_series_descriptions.csv", "Series descriptions")
df_coords      = load_csv_if_exists(RSNA_DATA_ROOT / "train_label_coordinates.csv",   "Label coordinates")

In [ ]:
# ── Distribution of severity labels across all conditions ──────────────────
if df_train is not None:
    condition_cols = [c for c in df_train.columns if c != "study_id"]
    severity_counts = (
        df_train[condition_cols]
        .apply(pd.Series.value_counts)
        .T
        .fillna(0)
        .astype(int)
    )
    fig, ax = plt.subplots(figsize=(14, 6))
    severity_counts.plot(kind="bar", ax=ax, colormap="RdYlGn_r", width=0.75)
    ax.set_title("Severity label distribution across all 25 condition-level combinations", fontsize=13)
    ax.set_xlabel("Condition × Level")
    ax.set_ylabel("Number of studies")
    ax.tick_params(axis="x", rotation=90)
    ax.legend(title="Severity")
    plt.tight_layout()
    plt.show()
else:
    print("Label distribution chart: requires train.csv")

# ── Series type distribution ────────────────────────────────────────────────
if df_series is not None:
    print("\nSeries description counts:")
    print(df_series["series_description"].value_counts().to_string())

---

## 🩻 DICOM loading and metadata inspection

The helper functions below demonstrate the core DICOM workflow:
1. Read a `.dcm` file with `pydicom.dcmread()`
2. Inspect the embedded metadata tags
3. Extract the raw pixel array and apply calibration

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────

def load_dicom(path: Path) -> pydicom.Dataset:
    """Load a DICOM file and return the dataset."""
    return pydicom.dcmread(str(path))


def print_dicom_metadata(ds: pydicom.Dataset) -> None:
    """Print a curated subset of clinically relevant DICOM tags."""
    tags_of_interest = [
        ("PatientID",              "Patient ID"),
        ("StudyDate",              "Study date"),
        ("Modality",               "Modality"),
        ("SeriesDescription",      "Series description"),
        ("BodyPartExamined",       "Body part"),
        ("StudyDescription",       "Study description"),
        ("Rows",                   "Image rows (px)"),
        ("Columns",                "Image cols (px)"),
        ("PixelSpacing",           "Pixel spacing (mm)"),
        ("SliceThickness",         "Slice thickness (mm)"),
        ("SpacingBetweenSlices",   "Slice spacing (mm)"),
        ("ImagePositionPatient",   "Image position (mm)"),
        ("ImageOrientationPatient","Image orientation"),
        ("WindowCenter",           "Window center"),
        ("WindowWidth",            "Window width"),
        ("RescaleSlope",           "Rescale slope"),
        ("RescaleIntercept",       "Rescale intercept"),
        ("BitsStored",             "Bits stored"),
        ("InstanceNumber",         "Instance number"),
    ]
    print(f"\n{'DICOM Metadata':=^55}")
    for attr, label in tags_of_interest:
        value = getattr(ds, attr, "— not present —")
        print(f"  {label:<30}: {value}")
    print(f"{'':=<55}")


def get_pixel_array(ds: pydicom.Dataset) -> np.ndarray:
    """
    Return calibrated pixel values.
    Applies RescaleSlope / RescaleIntercept if present.
    For MRI these are usually 1 / 0, but good practice to handle them.
    """
    pixels = ds.pixel_array.astype(np.float32)
    slope  = float(getattr(ds, "RescaleSlope",     1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return pixels * slope + intercept


print("DICOM helper functions defined.")

In [ ]:
# ── Find a sample DICOM file ───────────────────────────────────────────────
sample_dcm_path: Path | None = None

train_images_dir = RSNA_DATA_ROOT / "train_images"
if train_images_dir.exists():
    dcm_files = sorted(train_images_dir.rglob("*.dcm"))
    if dcm_files:
        sample_dcm_path = dcm_files[0]
        print(f"Found {len(dcm_files):,} DICOM files")
        print(f"Sample file: {sample_dcm_path}")
    else:
        print("No .dcm files found under train_images/")
else:
    print(f"train_images/ directory not found at {train_images_dir}")
    print("Download the dataset and update RSNA_DATA_ROOT to continue.")

# If running without dataset, point to any local .dcm you have for exploration:
# sample_dcm_path = Path("/path/to/your/file.dcm")

In [ ]:
if sample_dcm_path is not None:
    sample_ds = load_dicom(sample_dcm_path)
    print_dicom_metadata(sample_ds)
    pixels = get_pixel_array(sample_ds)
    print(f"\nPixel array shape : {pixels.shape}")
    print(f"Value range       : [{pixels.min():.0f}, {pixels.max():.0f}]")
    print(f"Mean / std        : {pixels.mean():.1f} / {pixels.std():.1f}")
else:
    print("No DICOM file available — skipping metadata display.")

---

## 🖼️ Window / Level normalization

Raw MRI pixel values span a wide dynamic range (e.g. 0–4095 for a 12-bit scan). To display the image meaningfully and isolate clinically relevant contrast, radiologists apply a **Window / Level** (W/L) transformation:

$$
I_{\text{display}} = \frac{I_{\text{raw}} - \left(\text{Level} - \dfrac{\text{Window}}{2}\right)}{\text{Window}} \times 255
$$

- **Level** (Window Center): the midpoint intensity of the range of interest
- **Window** (Window Width): the span of intensities mapped to the full 0–255 display range

Values below the lower bound become black; values above become white. This is how radiologists 
 when reading scans on a PACS workstation.

In [ ]:
def apply_window_level(
    pixels: np.ndarray,
    window: float | None = None,
    level: float | None = None,
) -> np.ndarray:
    """
    Apply window/level (W/L) contrast stretching and return an 8-bit array.
    If window and level are None, auto-scales using the image's own percentiles.
    """
    if window is None or level is None:
        low  = np.percentile(pixels, 1)
        high = np.percentile(pixels, 99)
        window = high - low
        level  = (high + low) / 2.0

    lower = level - window / 2.0
    upper = level + window / 2.0
    stretched = np.clip(pixels, lower, upper)
    stretched = (stretched - lower) / (upper - lower + 1e-8) * 255.0
    return stretched.astype(np.uint8)


def dicom_to_pil(ds: pydicom.Dataset, window: float | None = None, level: float | None = None) -> Image.Image:
    """
    Convert a DICOM dataset to a PIL Image with optional window/level control.
    Handles grayscale and RGB DICOM images.
    """
    pixels = get_pixel_array(ds)

    if pixels.ndim == 2:  # grayscale
        if window is not None and level is not None:
            # Use DICOM WindowCenter/Width if stored
            wc = float(getattr(ds, "WindowCenter", level))
            ww = float(getattr(ds, "WindowWidth",  window))
            if isinstance(wc, pydicom.sequence.Sequence):
                wc = float(wc[0])
                ww = float(ww[0])
            normed = apply_window_level(pixels, ww, wc)
        else:
            normed = apply_window_level(pixels)
        return Image.fromarray(normed, mode="L").convert("RGB")

    # RGB case
    normed = apply_window_level(pixels)
    return Image.fromarray(normed, mode="RGB")


print("Window/level helpers defined.")

In [ ]:
# ── Compare window/level presets on a single slice ──────────────────────────
if sample_dcm_path is not None:
    pixels = get_pixel_array(sample_ds)

    wl_presets = [
        ("Auto (1–99 percentile)", None,  None),
        ("Narrow window",          200,   400),
        ("Wide window",            2000,  1000),
    ]

    fig, axes = plt.subplots(1, len(wl_presets), figsize=(15, 5))
    fig.suptitle("Effect of Window/Level presets on MRI display", fontsize=13)

    for ax, (label, window, level) in zip(axes, wl_presets):
        normed = apply_window_level(pixels, window, level)
        ax.imshow(normed, cmap="gray")
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No DICOM file available — skipping window/level demo.")

---

## 📚 Visualizing a 3D MRI volume (multi-slice viewer)

Each MRI series consists of many parallel slices that together form a 3D volume.
The widget below lets you scroll through the stack interactively.

In [ ]:
def load_series_volume(
    series_dir: Path,
    max_slices: int = 60,
) -> tuple[list[np.ndarray], list[pydicom.Dataset]]:
    """
    Load all DICOM slices from a series directory, sorted by InstanceNumber.
    Returns (list_of_pixel_arrays, list_of_datasets).
    """
    dcm_files = sorted(series_dir.glob("*.dcm"))
    if not dcm_files:
        return [], []

    datasets = [pydicom.dcmread(str(f)) for f in dcm_files[:max_slices]]
    # Sort by InstanceNumber if available
    datasets.sort(key=lambda ds: int(getattr(ds, "InstanceNumber", 0)))
    pixel_arrays = [get_pixel_array(ds) for ds in datasets]
    return pixel_arrays, datasets


def show_series_contact_sheet(
    pixel_arrays: list[np.ndarray],
    series_description: str = "",
    cols: int = 6,
) -> None:
    """Display all slices of a series as a contact sheet."""
    n = len(pixel_arrays)
    if n == 0:
        print("No slices to display.")
        return
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = np.array(axes).flatten()
    fig.suptitle(f"{series_description}  ({n} slices)", fontsize=12)

    for idx, pixels in enumerate(pixel_arrays):
        normed = apply_window_level(pixels)
        axes[idx].imshow(normed, cmap="gray")
        axes[idx].set_title(f"#{idx+1}", fontsize=7)
        axes[idx].axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


print("Volume loading helpers defined.")

In [ ]:
# ── Load and display a full series volume ───────────────────────────────────
selected_series_dir: Path | None = None

if df_series is not None and train_images_dir.exists():
    # Pick the first Sagittal T2/STIR series found in the dataset
    t2_rows = df_series[df_series["series_description"].str.contains("Sagittal T2", case=False, na=False)]
    if not t2_rows.empty:
        row       = t2_rows.iloc[0]
        study_id  = str(row["study_id"])
        series_id = str(row["series_id"])
        selected_series_dir = train_images_dir / study_id / series_id
        print(f"Loading series: {row['series_description']}")
        print(f"  Study ID : {study_id}")
        print(f"  Series ID: {series_id}")
        print(f"  Path     : {selected_series_dir}")
    else:
        print("No Sagittal T2 series found — trying first available series.")
        row       = df_series.iloc[0]
        study_id  = str(row["study_id"])
        series_id = str(row["series_id"])
        selected_series_dir = train_images_dir / study_id / series_id
elif sample_dcm_path is not None:
    selected_series_dir = sample_dcm_path.parent
    print(f"Using parent directory of sample file: {selected_series_dir}")
else:
    print("No DICOM data available.")


# Load and display
if selected_series_dir is not None and selected_series_dir.exists():
    pixel_arrays, datasets = load_series_volume(selected_series_dir, max_slices=30)
    series_desc = getattr(datasets[0], "SeriesDescription", "Unknown series") if datasets else ""
    print(f"\nLoaded {len(pixel_arrays)} slices. Series: {series_desc}")
    show_series_contact_sheet(pixel_arrays, series_description=series_desc, cols=6)
else:
    print("Directory not found — skipping volume visualization.")

In [ ]:
# ── Interactive slice scroller (requires ipywidgets) ────────────────────────
if widgets_available and selected_series_dir is not None and selected_series_dir.exists():
    pixel_arrays, datasets = load_series_volume(selected_series_dir, max_slices=60)

    slider = widgets.IntSlider(
        value=len(pixel_arrays) // 2,
        min=0, max=len(pixel_arrays) - 1,
        step=1,
        description="Slice:",
        continuous_update=True,
        layout=widgets.Layout(width="500px"),
    )
    out = widgets.Output()

    def _update_slice(change):
        idx = change["new"]
        with out:
            clear_output(wait=True)
            ds  = datasets[idx]
            px  = get_pixel_array(ds)
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.imshow(apply_window_level(px), cmap="gray")
            ax.set_title(
                f"Slice {idx + 1}/{len(pixel_arrays)}  "
                f"— {getattr(ds, 'SeriesDescription', '')}",
                fontsize=10,
            )
            ax.axis("off")
            plt.tight_layout()
            plt.show()

    slider.observe(_update_slice, names="value")
    _update_slice({"new": slider.value})
    display(widgets.VBox([slider, out]))
elif not widgets_available:
    print("ipywidgets not available — install it for the interactive scroller.")
else:
    print("No DICOM data loaded — skipping interactive scroller.")

---

## 🤖 MedGemma multimodal API helpers

The functions below mirror the pattern used in `medical_image_analysis.ipynb`:
1. Encode an image (PIL or DICOM) as a Base64 data URL
2. Build the OpenAI-compatible message list with an image and text
3. POST to the LM Studio `/chat/completions` endpoint

In [ ]:
def pil_to_data_url(image: Image.Image, fmt: str = "PNG") -> str:
    """Encode a PIL Image as a Base64 data URL (PNG by default)."""
    buf = io.BytesIO()
    image.save(buf, format=fmt)
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    mime = f"image/{fmt.lower()}"
    return f"data:{mime};base64,{encoded}"


def file_to_data_url(path: Path) -> str:
    """Encode any image file as a Base64 data URL."""
    mime_type, _ = mimetypes.guess_type(path.name)
    mime_type = mime_type or "image/png"
    with path.open("rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"


def dicom_to_data_url(ds: pydicom.Dataset) -> str:
    """Convert a DICOM dataset to a Base64-encoded PNG data URL for the API."""
    pil_img = dicom_to_pil(ds)
    return pil_to_data_url(pil_img)


def build_messages(system_prompt: str, user_prompt: str, image_data_urls: list[str]) -> list[dict]:
    """
    Build an OpenAI-compatible messages list for a multimodal request.
    Supports multiple images in a single user turn.
    """
    content = [{"type": "text", "text": user_prompt}]
    for url in image_data_urls:
        content.append({"type": "image_url", "image_url": {"url": url}})

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": content},
    ]


def call_medgemma(
    messages: list[dict],
    max_tokens: int = 800,
    temperature: float = 0.2,
) -> str:
    """Send a request to the LM Studio MedGemma endpoint and return the reply text."""
    payload = {
        "model":       LM_STUDIO_MODEL,
        "messages":    messages,
        "temperature": temperature,
        "max_tokens":  max_tokens,
    }
    response = requests.post(
        f"{LM_STUDIO_BASE_URL}/chat/completions",
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    if isinstance(content, list):
        content = "\n".join(p.get("text", "") for p in content if p.get("type") == "text").strip()
    return content


SYSTEM_PROMPT = (
    "You are an expert neuroradiologist AI assistant specializing in lumbar spine MRI interpretation. "
    "Produce concise, clinician-facing reports. Use precise anatomical terminology. "
    "Always state that AI findings must be confirmed by a licensed radiologist before clinical action."
)

print("MedGemma API helpers defined.")

---

## 🧠 MedGemma Analysis 1 — Sagittal T2/STIR MRI

**Sagittal T2/STIR** is the most informative sequence for:
- Disc dehydration (dark discs = degeneration)
- Spinal canal stenosis (compression of the bright CSF column)
- Vertebral bone marrow edema (bright signal in Modic changes)
- Overall spinal alignment (lordosis / kyphosis)

The prompt below asks MedGemma to grade each disc level using the standard **5-point Pfirrmann grading** scale and assess spinal canal stenosis.

In [ ]:
SAGITTAL_T2_PROMPT = """
You are reviewing a sagittal T2-weighted (or T2/STIR) lumbar spine MRI image.

Produce a structured radiological report with the following sections:

1. Overall alignment
   - Describe lumbar lordosis (normal / reduced / reversed).
   - Note any spondylolisthesis or listhesis.

2. Intervertebral disc assessment (L1/L2 through L5/S1)
   For each level report:
   - Disc signal intensity (bright = normal hydration; dark = degeneration)
   - Disc height (maintained / reduced / collapsed)
   - Pfirrmann grade if determinable (I–V)
   - Posterior disc contour (normal / bulge / protrusion / extrusion)

3. Spinal canal
   - Assess calibre of the central canal at each level.
   - Grade spinal canal stenosis per level: None / Mild / Moderate / Severe

4. Vertebral bodies
   - Note any compression fractures, Modic endplate changes, or bone marrow signal abnormalities.

5. Summary
   - Identify the one or two levels most likely to be clinically significant.
   - Suggest the most appropriate additional sequence or intervention.

6. Disclaimer
   - State that this is AI-generated decision support and must be reviewed by a licensed radiologist.

Keep the tone professional and medically precise. Do not overstate certainty.
""".strip()

print(SAGITTAL_T2_PROMPT)

In [ ]:
# ── Pick the middle slice of the Sagittal T2 series for analysis ─────────
sagittal_t2_pil: Image.Image | None = None

if selected_series_dir is not None and selected_series_dir.exists() and datasets:
    mid_idx = len(datasets) // 2
    sagittal_t2_ds  = datasets[mid_idx]
    sagittal_t2_pil = dicom_to_pil(sagittal_t2_ds)

    print(f"Using slice {mid_idx + 1}/{len(datasets)} from series:")
    print(f"  {getattr(sagittal_t2_ds, 'SeriesDescription', 'Unknown')}")
    print(f"  Instance number: {getattr(sagittal_t2_ds, 'InstanceNumber', '?')}")
    print(f"  Image size: {sagittal_t2_pil.size[0]} × {sagittal_t2_pil.size[1]} px")
    display(sagittal_t2_pil)
else:
    print("No DICOM data loaded — cannot run MedGemma analysis.")
    print("Download the RSNA dataset and re-run the data loading cells.")

In [ ]:
# ── Send the sagittal T2 slice to MedGemma ───────────────────────────────
if sagittal_t2_pil is not None:
    data_url  = pil_to_data_url(sagittal_t2_pil)
    messages  = build_messages(SYSTEM_PROMPT, SAGITTAL_T2_PROMPT, [data_url])

    print("Sending image to MedGemma … (this may take 30–120 s)")
    reply = call_medgemma(messages, max_tokens=900)

    series_desc = getattr(sagittal_t2_ds, "SeriesDescription", "Sagittal T2")
    study_id_tag = getattr(sagittal_t2_ds, "StudyID", "unknown")

    report_md = "\n".join([
        "## MedGemma Report — Sagittal T2/STIR Lumbar Spine MRI",
        "",
        f"**Series:** `{series_desc}`  |  **Study ID:** `{study_id_tag}`  |  **Slice:** {mid_idx + 1}/{len(datasets)}",
        "",
        reply,
    ])
    display(Markdown(report_md))
else:
    print("No image to send — skipping MedGemma call.")

---

## 🧠 MedGemma Analysis 2 — Axial T2 MRI

**Axial T2** images provide a cross-sectional view of the spine at each disc level and are the gold standard for assessing:
- **Central canal stenosis** — compression of the thecal sac
- **Lateral recess (subarticular) stenosis** — compression before the nerve root enters the foramen
- **Neural foraminal stenosis** — compression of the exiting nerve root in the foramen

The grading used in the RSNA competition maps directly to these axial findings.

In [ ]:
# ── Load an Axial T2 series ───────────────────────────────────────────────
axial_series_dir: Path | None = None
axial_pixel_arrays: list[np.ndarray] = []
axial_datasets: list[pydicom.Dataset] = []

if df_series is not None and train_images_dir.exists():
    axial_rows = df_series[df_series["series_description"].str.contains("Axial T2", case=False, na=False)]
    if not axial_rows.empty:
        # Find a study that also has a Sagittal T2 (for comparison)
        if "study_id" in df_series.columns:
            sag_study_ids = set(
                df_series[df_series["series_description"].str.contains("Sagittal T2", case=False, na=False)]["study_id"]
            )
            matched = axial_rows[axial_rows["study_id"].isin(sag_study_ids)]
            row = matched.iloc[0] if not matched.empty else axial_rows.iloc[0]
        else:
            row = axial_rows.iloc[0]

        axial_series_dir = train_images_dir / str(row["study_id"]) / str(row["series_id"])
        print(f"Axial series: {row['series_description']}")
        print(f"  Study ID : {row['study_id']}")
        print(f"  Series ID: {row['series_id']}")

        if axial_series_dir.exists():
            axial_pixel_arrays, axial_datasets = load_series_volume(axial_series_dir, max_slices=25)
            show_series_contact_sheet(
                axial_pixel_arrays,
                series_description=row["series_description"],
                cols=5,
            )
        else:
            print(f"Directory not found: {axial_series_dir}")
    else:
        print("No Axial T2 series found in the dataset.")
else:
    print("Series descriptions CSV not loaded — skipping Axial T2 selection.")

In [ ]:
AXIAL_T2_PROMPT = """
You are reviewing an axial T2-weighted lumbar spine MRI image (cross-sectional view).

This image shows the spine as if looking down from above at a single disc or vertebral level.

Produce a structured radiological report with the following sections:

1. Disc level (if identifiable)
   - State which disc level this axial image represents (e.g., L4/L5).

2. Central canal
   - Describe the shape and calibre of the spinal canal.
   - Assess thecal sac compression: None / Mild / Moderate / Severe
   - Describe the CSF signal surrounding the cauda equina nerve roots.

3. Lateral recesses (subarticular zones)
   - Left lateral recess: patent / stenotic (grade: None / Mild / Moderate / Severe)
   - Right lateral recess: patent / stenotic (grade: None / Mild / Moderate / Severe)
   - Note if nerve roots appear compressed or displaced.

4. Neural foramina
   - Left foramen: patent / narrowed (grade: None / Mild / Moderate / Severe)
   - Right foramen: patent / narrowed (grade: None / Mild / Moderate / Severe)
   - Note any perineural fat obliteration or nerve root contact with disc/osteophyte.

5. Facet joints
   - Describe articular facet appearance: normal / hypertrophied / arthritic
   - Note any ligamentum flavum thickening or buckling.

6. Posterior elements
   - Describe the posterior paraspinal muscles if visible.

7. Summary
   - Identify dominant pathology (e.g., central stenosis vs. foraminal narrowing).
   - State the predominant side if lateralised.

8. Disclaimer
   - State that this is AI-generated decision support and must be reviewed by a licensed radiologist.
""".strip()

print(AXIAL_T2_PROMPT)

In [ ]:
# ── Send a representative axial slice to MedGemma ───────────────────────
axial_t2_pil: Image.Image | None = None

if axial_datasets:
    mid_axial_idx = len(axial_datasets) // 2
    axial_ds  = axial_datasets[mid_axial_idx]
    axial_t2_pil = dicom_to_pil(axial_ds)

    print(f"Using axial slice {mid_axial_idx + 1}/{len(axial_datasets)}")
    display(axial_t2_pil)

    data_url = pil_to_data_url(axial_t2_pil)
    messages = build_messages(SYSTEM_PROMPT, AXIAL_T2_PROMPT, [data_url])

    print("\nSending axial image to MedGemma … (this may take 30–120 s)")
    reply = call_medgemma(messages, max_tokens=900)

    series_desc = getattr(axial_ds, "SeriesDescription", "Axial T2")
    study_id_tag = getattr(axial_ds, "StudyID", "unknown")

    report_md = "\n".join([
        "## MedGemma Report — Axial T2 Lumbar Spine MRI",
        "",
        f"**Series:** `{series_desc}`  |  **Study ID:** `{study_id_tag}`  |  **Slice:** {mid_axial_idx + 1}/{len(axial_datasets)}",
        "",
        reply,
    ])
    display(Markdown(report_md))
else:
    print("No axial DICOM data loaded — skipping MedGemma axial analysis.")

---

## 🧠 MedGemma Analysis 3 — Multi-series contextual report

The most powerful use of a multimodal model is sending **multiple images from the same study** in a single request, allowing the model to reason across different planes and sequences — just as a radiologist does.

Here we send three images simultaneously:
- **Sagittal T2** (mid-sagittal) — overall canal and disc overview
- **Axial T2** (at the most stenotic level) — detailed cross-sectional grading
- **Sagittal T1** (if available) — vertebral body and bone marrow assessment

In [ ]:
def load_series_by_type(
    df_series: pd.DataFrame,
    study_id: str,
    description_keyword: str,
    train_images_dir: Path,
) -> list[pydicom.Dataset]:
    """Load the DICOM datasets for a specific series type within a study."""
    rows = df_series[
        (df_series["study_id"].astype(str) == study_id) &
        (df_series["series_description"].str.contains(description_keyword, case=False, na=False))
    ]
    if rows.empty:
        return []
    series_id = str(rows.iloc[0]["series_id"])
    series_dir = train_images_dir / study_id / series_id
    if not series_dir.exists():
        return []
    _, dsets = load_series_volume(series_dir, max_slices=60)
    return dsets


print("Multi-series loader defined.")

In [ ]:
MULTI_SERIES_PROMPT = """
You are reviewing three lumbar spine MRI images from the same patient study:
  Image 1: Sagittal T2/STIR (side view — assess canal and discs)
  Image 2: Axial T2 (cross-section — assess foraminal and lateral recess stenosis)
  Image 3: Sagittal T1 (side view — assess bone marrow and alignment)

Cross-reference all three images to produce a comprehensive radiological report:

1. Summary of findings
   - List the dominant abnormalities identified across all sequences.

2. Disc-level grading (L1/L2 to L5/S1)
   For each level, synthesise sagittal and axial information to grade:
   - Spinal canal stenosis: Normal/Mild | Moderate | Severe
   - Left neural foraminal narrowing: Normal/Mild | Moderate | Severe
   - Right neural foraminal narrowing: Normal/Mild | Moderate | Severe
   - Left subarticular stenosis: Normal/Mild | Moderate | Severe
   - Right subarticular stenosis: Normal/Mild | Moderate | Severe

3. Most clinically significant level
   - Identify the level most likely to explain the patient's symptoms.
   - Explain which image (sagittal or axial) provides the most critical evidence.

4. Differential diagnoses
   - List alternative interpretations if findings are non-specific.

5. Recommended clinical action
   - Suggest whether conservative management, further imaging, or surgical consultation is appropriate.

6. Disclaimer
   - State that this AI-generated report must be verified by a board-certified radiologist.
""".strip()

print(MULTI_SERIES_PROMPT)

In [ ]:
# ── Collect images from all three series for the same study ─────────────
multi_images: list[Image.Image] = []
multi_labels: list[str] = []

if df_series is not None and train_images_dir.exists() and sagittal_t2_pil is not None:
    # Use the same study as the sagittal T2 loaded earlier
    target_study_id = str(getattr(sagittal_t2_ds, "StudyID",
                           selected_series_dir.parent.name if selected_series_dir else "unknown"))

    # Try to infer study_id from the directory path if StudyID tag isn't set
    if target_study_id == "unknown" and selected_series_dir is not None:
        target_study_id = selected_series_dir.parent.name

    for keyword, label in [("Sagittal T2", "Sagittal T2/STIR"), ("Axial T2", "Axial T2"), ("Sagittal T1", "Sagittal T1")]:
        dsets = load_series_by_type(df_series, target_study_id, keyword, train_images_dir)
        if dsets:
            mid = len(dsets) // 2
            pil_img = dicom_to_pil(dsets[mid])
            multi_images.append(pil_img)
            multi_labels.append(label)
            print(f"  Loaded: {label} ({len(dsets)} slices, using slice {mid+1})")
        else:
            print(f"  Not found: {label}")

    if multi_images:
        # Display selected slices side-by-side
        fig, axes = plt.subplots(1, len(multi_images), figsize=(6 * len(multi_images), 7))
        if len(multi_images) == 1:
            axes = [axes]
        for ax, img, lbl in zip(axes, multi_images, multi_labels):
            ax.imshow(np.array(img.convert("L")), cmap="gray")
            ax.set_title(lbl, fontsize=11)
            ax.axis("off")
        fig.suptitle(f"Multi-series input to MedGemma  |  Study: {target_study_id}", fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("No multi-series images collected.")
else:
    print("Multi-series analysis requires the RSNA dataset and loaded sagittal series.")

In [ ]:
# ── Send all series images to MedGemma in one request ────────────────────
if multi_images:
    data_urls = [pil_to_data_url(img) for img in multi_images]
    messages  = build_messages(SYSTEM_PROMPT, MULTI_SERIES_PROMPT, data_urls)

    print(f"Sending {len(multi_images)} images to MedGemma … (this may take 60–180 s)")
    reply = call_medgemma(messages, max_tokens=1200)

    report_md = "\n".join([
        "## MedGemma Multi-Series Lumbar Spine Report",
        "",
        f"**Series submitted:** {' | '.join(multi_labels)}",
        f"**Study:** `{target_study_id}`",
        "",
        reply,
    ])
    display(Markdown(report_md))
else:
    print("No images collected for multi-series analysis.")

---

## 📊 Label-aligned analysis: overlay ground-truth coordinates on the image

The competition provides `train_label_coordinates.csv` which contains `(x, y)` pixel coordinates marking the center of each annotated condition at each level.
This cell overlays those annotations on the sagittal T2 image and then asks MedGemma to specifically comment on the marked regions.

In [ ]:
def overlay_label_coordinates(
    image: Image.Image,
    coords_df: pd.DataFrame,
    study_id: str,
    series_id: str,
    instance_number: int,
) -> Image.Image:
    """
    Draw RSNA label coordinate annotations onto a PIL image and return the annotated image.
    """
    relevant = coords_df[
        (coords_df["study_id"].astype(str)  == str(study_id))  &
        (coords_df["series_id"].astype(str) == str(series_id)) &
        (coords_df["instance_number"].astype(str) == str(instance_number))
    ]
    if relevant.empty:
        return image  # No annotations for this slice

    condition_colors = {
        "spinal_canal_stenosis":          "yellow",
        "left_neural_foraminal_narrowing": "cyan",
        "right_neural_foraminal_narrowing": "magenta",
        "left_subarticular_stenosis":      "lime",
        "right_subarticular_stenosis":     "orange",
    }

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(np.array(image.convert("L")), cmap="gray")

    for _, row in relevant.iterrows():
        cond   = row.get("condition", "")
        level  = row.get("level", "")
        x, y   = float(row["x"]), float(row["y"])
        color  = condition_colors.get(cond, "red")
        circle = plt.Circle((x, y), radius=8, color=color, fill=False, linewidth=2)
        ax.add_patch(circle)
        ax.text(x + 10, y, f"{cond.replace('_', ' ')}\n{level}",
                color=color, fontsize=6, va="center")

    # Legend
    patches = [mpatches.Patch(color=c, label=k.replace("_", " "))
               for k, c in condition_colors.items()]
    ax.legend(handles=patches, loc="upper right", fontsize=6, framealpha=0.7)
    ax.set_title(f"Label annotations — slice {instance_number}", fontsize=10)
    ax.axis("off")
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="PNG", dpi=120, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()


print("Annotation overlay helper defined.")

In [ ]:
# ── Find and display an annotated slice ─────────────────────────────────
annotated_image: Image.Image | None = None

if df_coords is not None and datasets and selected_series_dir is not None:
    study_id_folder  = selected_series_dir.parent.name
    series_id_folder = selected_series_dir.name

    relevant_coords = df_coords[
        (df_coords["study_id"].astype(str)  == study_id_folder) &
        (df_coords["series_id"].astype(str) == series_id_folder)
    ]

    if relevant_coords.empty:
        print(f"No annotations for study={study_id_folder}, series={series_id_folder}")
        print("Trying a study that has annotations …")
        # Find a study+series combination that has annotations
        if not df_coords.empty:
            first_coord = df_coords.iloc[0]
            study_id_folder  = str(first_coord["study_id"])
            series_id_folder = str(first_coord["series_id"])
            relevant_coords = df_coords[
                (df_coords["study_id"].astype(str)  == study_id_folder) &
                (df_coords["series_id"].astype(str) == series_id_folder)
            ]
            annotated_series_dir = train_images_dir / study_id_folder / series_id_folder
            if annotated_series_dir.exists():
                _, annotated_datasets = load_series_volume(annotated_series_dir, max_slices=60)
            else:
                annotated_datasets = []
        else:
            annotated_datasets = []
    else:
        annotated_datasets = datasets
        annotated_series_dir = selected_series_dir

    if not relevant_coords.empty and annotated_datasets:
        # Pick the instance number that has the most annotations
        best_instance = int(
            relevant_coords.groupby("instance_number").size().idxmax()
        )
        matching_ds = next(
            (ds for ds in annotated_datasets
             if int(getattr(ds, "InstanceNumber", -1)) == best_instance),
            annotated_datasets[len(annotated_datasets) // 2]
        )
        pil_slice = dicom_to_pil(matching_ds)
        annotated_image = overlay_label_coordinates(
            pil_slice, df_coords,
            study_id_folder, series_id_folder,
            int(getattr(matching_ds, "InstanceNumber", best_instance)),
        )
        print(f"Annotated instance {best_instance} with {len(relevant_coords)} annotations")
        display(annotated_image)
    else:
        print("No annotated slices found in the loaded data.")
else:
    print("Label coordinates CSV not loaded — skipping annotation overlay.")

---

## 📝 Summary: comparing the three MedGemma analysis approaches

| Approach | Images sent | Prompt focus | Best for |
| :--- | :--- | :--- | :--- |
| **Sagittal T2 single** | 1 (mid-sagittal) | Overall canal + disc grading | Quick overall impression |
| **Axial T2 single** | 1 (representative axial) | Foraminal and subarticular detail | Lateralised nerve compression |
| **Multi-series** | 3 (sag T1 + T2 + axial T2) | Full 25-condition severity grid | Most comprehensive diagnosis |

### Key takeaways

1. **DICOM is richer than PNG/JPEG** — always extract raw pixel values and apply window/level before sending to a vision model. The default JPEG encoding would lose the 12-bit dynamic range.

2. **Prompt engineering matters** — providing structured section headings and explicitly naming the five RSNA conditions gives the model the vocabulary to produce actionable, structured output.

3. **Multi-image context** dramatically improves localization — the model can cross-reference the sagittal overview with the axial detail, mimicking how a real radiologist reads a spine study.

4. **MedGemma vs. general LLMs** — because MedGemma is fine-tuned on medical images and text, it understands terms like "Pfirrmann grade", "ligamentum flavum buckling", and "perineural fat obliteration" without requiring explanations.

5. **Always include a disclaimer** — AI-generated radiological reports are decision-support tools only; they must be reviewed and confirmed by a licensed radiologist before influencing clinical decisions.

---

### Further exploration
- Use `train_label_coordinates.csv` to crop tightly around each disc level and send individual crops for per-level severity prediction.
- Compare MedGemma's qualitative severity grading against the ground-truth `Normal/Mild / Moderate / Severe` labels in `train.csv`.
- Extend the notebook to use the full volume (all slices of each series) by sending a montage image constructed from 9 representative slices.
- Try few-shot prompting: prepend 2–3 examples of graded slices with known labels to guide the model toward the competition's exact grading rubric.